#Schritt 2: Vorverarbeitung

Songtexte bereinigen

In [ ]:
import pandas as pd
import re
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('tokenizers/punkt_tab')
except LookupError:
    nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize

In [ ]:
def entferne_strophen_markierungen(text):
    ohne_marker = re.sub('\\[.*?\\]', ' ', text)
    return ohne_marker

In [ ]:
def bereinige_text(text):
    text = text.lower()
    text = re.sub('[^a-z0-9\\s]', ' ', text)
    text = re.sub('\\s+', ' ', text).strip()
    return text

In [ ]:
def tokenisiere(text):
    # Wird aktuell von keinem der weiteren Schritte gebraucht.
    tokens = word_tokenize(text, language='english')
    return tokens

In [ ]:
def verarbeite_zeile(roh_text):
    ohne_marker = entferne_strophen_markierungen(roh_text)
    sauber = bereinige_text(ohne_marker)
    return sauber

In [ ]:
def teile_in_abschnitte(roh_text):
    marker_liste = re.findall('\\[(.*?)\\]', roh_text)
    if len(marker_liste) == 0:
        return [('Ganzer_Text', roh_text)]
    teile = re.split('\\[(.*?)\\]', roh_text)
    abschnitte = []
    i = 1
    while i < len(teile) - 1:
        abschnitts_typ = teile[i].strip()
        abschnitts_text = teile[i + 1].strip()
        if abschnitts_text != '':
            abschnitte.append((abschnitts_typ, abschnitts_text))
        i += 2
    if len(abschnitte) == 0:
        return [('Ganzer_Text', roh_text)]
    return abschnitte

In [ ]:
def erstelle_segment_tabelle(daten):
    segment_liste = []
    for index, zeile in daten.iterrows():
        abschnitte = teile_in_abschnitte(zeile['lyrics'])
        segment_nummer = 1
        for abschnitts_typ, abschnitts_text in abschnitte:
            sauberer_text = bereinige_text(abschnitts_text)
            segment_liste.append({'song_id': zeile['song_id'], 'title': zeile['title'], 'playlist_id': zeile['playlist_id'] if 'playlist_id' in daten.columns else None, 'segment_nr': segment_nummer, 'segment_typ': abschnitts_typ, 'segment_text_clean': sauberer_text})
            segment_nummer += 1
    segment_tabelle = pd.DataFrame(segment_liste)
    return segment_tabelle

In [ ]:
def verarbeite_datensatz(daten):
    saubere_texte = []
    for index, zeile in daten.iterrows():
        roh_text = zeile['lyrics']
        sauberer_text = verarbeite_zeile(roh_text)
        saubere_texte.append(sauberer_text)
    daten['lyrics_clean'] = saubere_texte
    return daten

#Pipeline ausführen

In [ ]:
daten = pd.read_csv('data/rohdaten.csv')

#Song-Ebene

In [ ]:
daten = verarbeite_datensatz(daten)
print('Song-Ebene')
print(daten[['title', 'lyrics_clean']].head())
daten.to_csv('data/daten_vorverarbeitet.csv', index=False)
print('Vorverarbeitete Daten (Song-Ebene) gespeichert unter: data/daten_vorverarbeitet.csv')

Abschnitts-Ebene

In [ ]:
daten_roh_fuer_segmente = pd.read_csv('data/rohdaten.csv')
segment_tabelle = erstelle_segment_tabelle(daten_roh_fuer_segmente)
print('\nAbschnitts-Ebene (Verse/Chorus)')
print(segment_tabelle[['title', 'segment_nr', 'segment_typ', 'segment_text_clean']])
segment_tabelle.to_csv('data/segmente_vorverarbeitet.csv', index=False)
print('Vorverarbeitete Daten (Abschnitts-Ebene) gespeichert unter: data/segmente_vorverarbeitet.csv')